In [1]:
import torch
from pathlib import Path

# CHANGE THIS to the exact Qwen3 1.5B model repo
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # your actual model id

DATA_FILE = "spider_style_dataset.jsonl"
OUTPUT_DIR = "qwen_spider_full_ft"

MAX_SEQ_LEN = 512
BATCH_SIZE = 1      # full fine-tuning → must keep small
GRAD_ACCUM_STEPS = 16  # effective batch = 16
LR = 1e-5                 # lower LR for full FT (LoRA uses higher LR)
NUM_EPOCHS = 3
WARMUP_RATIO = 0.03

device = "cuda"
device


'cuda'

In [2]:
from datasets import load_dataset

raw_ds = load_dataset("json", data_files=DATA_FILE, split="train")
print(raw_ds)
print(raw_ds[0])


Dataset({
    features: ['messages'],
    num_rows: 2503
})
{'messages': [{'role': 'system', 'content': 'You are a text-to-SQL expert.'}, {'role': 'user', 'content': 'Given schema:\nCREATE TABLE journals (\n  journal_id INTEGER PRIMARY KEY,\n  name TEXT,\n  publisher TEXT,\n  issn TEXT,\n  impact_factor REAL\n);\n\nCREATE TABLE articles (\n  article_id INTEGER PRIMARY KEY,\n  title TEXT,\n  pub_date DATE,\n  journal_id INTEGER\n);\n\nCREATE TABLE authors (\n  author_id INTEGER PRIMARY KEY,\n  name TEXT,\n  affiliation TEXT,\n  country TEXT\n);\n\nCREATE TABLE article_authors (\n  aa_id INTEGER PRIMARY KEY,\n  article_id INTEGER,\n  author_id INTEGER\n);\n\nCREATE TABLE citations (\n  citation_id INTEGER PRIMARY KEY,\n  citing_article_id INTEGER,\n  cited_article_id INTEGER,\n  citation_year INTEGER\n);\n\nQuestion: What is the average impact factor of all journals?'}, {'role': 'assistant', 'content': 'SELECT AVG(impact_factor) FROM journals;'}]}


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model


`torch_dtype` is deprecated! Use `dtype` instead!


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [5]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

ds = raw_ds.map(format_example)
print(ds[0]["text"][:500])


<|im_start|>system
You are a text-to-SQL expert.<|im_end|>
<|im_start|>user
Given schema:
CREATE TABLE journals (
  journal_id INTEGER PRIMARY KEY,
  name TEXT,
  publisher TEXT,
  issn TEXT,
  impact_factor REAL
);

CREATE TABLE articles (
  article_id INTEGER PRIMARY KEY,
  title TEXT,
  pub_date DATE,
  journal_id INTEGER
);

CREATE TABLE authors (
  author_id INTEGER PRIMARY KEY,
  name TEXT,
  affiliation TEXT,
  country TEXT
);

CREATE TABLE article_authors (
  aa_id INTEGER PRIMARY KEY,
 


In [6]:
def tokenize(batch):
    return tokenizer(
        batch["text"], 
        truncation=True, 
        max_length=MAX_SEQ_LEN, 
        padding="max_length"
    )

tokenized_ds = ds.map(tokenize, batched=True, remove_columns=ds.column_names)
tokenized_ds


Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2503
})

In [7]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Data collator: standard causal LM
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# IMPORTANT: enable checkpointing before creating the Trainer
model.gradient_checkpointing_enable()
model.config.use_cache = False

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,   # smallest per-step batch to save VRAM
    gradient_accumulation_steps=48,  # effective batch = 48
    num_train_epochs=NUM_EPOCHS,
    learning_rate=5e-6,              # safer LR for full FT
    warmup_ratio=WARMUP_RATIO,
    logging_steps=10,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,               # gradient clipping → prevents huge loss
    report_to="none",
    # you can drop `optim` so HF picks a good default;
    # if you want explicit: optim="adamw_torch_fused" (if your version supports it)
)

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=tokenized_ds,
    data_collator=data_collator,
)


/tmp/ipykernel_4197/1501605503.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The model is already on multiple devices. Skipping the move to device specified in `args`.


In [8]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,16.624400
20,38.665600
30,35.429500
40,28.745700
50,39.173000
60,51.603100
70,31.524200
80,28.264400
90,21.088100
100,334.111800


TrainOutput(global_step=159, training_loss=66.92917676241893, metrics={'train_runtime': 2359.1919, 'train_samples_per_second': 3.183, 'train_steps_per_second': 0.067, 'total_flos': 3.0226475905449984e+16, 'train_loss': 66.92917676241893, 'epoch': 3.0})

In [9]:
from pathlib import Path

OUTPUT_DIR = "qwen_spider_full_ft"

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved fine-tuned model to", Path(OUTPUT_DIR).resolve())


Saved fine-tuned model to /teamspace/studios/this_studio/qwen_spider_full_ft
